# Recommender System Modeling

This notebook implements and evaluates three recommendation approaches:

- Collaborative Filtering using Matrix Factorization
- Content-Based Filtering using TF-IDF features
- A Hybrid recommender combining collaborative and content-based scores

The models are evaluated using both a random train/validation/test split and a chronological temporal split to provide a more realistic estimate of future recommendation performance.

## Imports

In [ ]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Database connection

In [ ]:
server = "."
database = "MovieRecommenderDB"

connection_string = (
    f"mssql+pyodbc://{server}/{database}"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
)
engine = create_engine(connection_string)

## Load ratings

In [ ]:
ratings_query = """
SELECT
    user_id,
    movie_id,
    rating
FROM dbo.Ratings;
"""
ratings = pd.read_sql(ratings_query, con=engine)
print(ratings.shape)
ratings.head()

## Train / Validation / Test Split

The ratings data is divided into training, validation, and test sets.

The random split is used as a benchmark for comparing the recommendation approaches under a standard offline evaluation setting.

In [ ]:
train_ratings, temp_ratings = train_test_split(ratings, test_size=0.2, random_state=42)

In [ ]:
validation_ratings, test_ratings = train_test_split(temp_ratings, test_size=0.5, random_state=42)

In [ ]:
print("Train: ", train_ratings.shape)
print("Test: ", test_ratings.shape)
print("validation: ", validation_ratings.shape)

In [ ]:
user_ids = ratings["user_id"].unique()
movie_ids = ratings["movie_id"].unique()

## ID mapping

In [ ]:
user_id_to_index = {
    user_id : index
    for index, user_id in enumerate(user_ids)
}

movie_id_to_index = {
    movie_id : index
    for index, movie_id in enumerate(movie_ids)
}

## Baseline Model

A user-mean rating baseline is used as a simple reference point for rating prediction.

The baseline provides a comparison for determining whether the Matrix Factorization model improves upon a simple non-personalized prediction strategy.

In [ ]:
user_mean_rating = (train_ratings.groupby("user_id")["rating"].mean())
user_mean_rating.head()

In [ ]:
validation_predictions = validation_ratings['user_id'].map(user_mean_rating)
validation_predictions = validation_predictions.fillna(train_ratings["rating"].mean())
validation_predictions[0:5]

In [ ]:
baseline_mae = mean_absolute_error(validation_ratings["rating"], validation_predictions)

baseline_rmse = np.sqrt(mean_squared_error(validation_ratings["rating"],validation_predictions))

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)

## Collaborative Filtering — Matrix Factorization

Collaborative Filtering is implemented using Matrix Factorization.

Each user and movie is represented by a learnable latent vector. The predicted rating is computed from the interaction between the user and movie latent representations, together with user and movie bias terms.

In [ ]:
class MatrixFactorization(nn.Module):

    def __init__(
        self,
        num_users,
        num_movies,
        global_mean,
        embedding_dim=50
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.movie_embedding = nn.Embedding(
            num_movies,
            embedding_dim
        )

        self.user_bias = nn.Embedding(
            num_users,
            1
        )

        self.movie_bias = nn.Embedding(
            num_movies,
            1
        )

        self.global_bias = nn.Parameter(
            torch.tensor(global_mean, dtype=torch.float32)
        )

        # Small random initialization for latent factors
        nn.init.normal_(
            self.user_embedding.weight,
            mean=0.0,
            std=0.01
        )

        nn.init.normal_(
            self.movie_embedding.weight,
            mean=0.0,
            std=0.01
        )

        # Start user/movie biases at zero
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)


    def forward(self, users, movies):

        user_vector = self.user_embedding(users)
        movie_vector = self.movie_embedding(movies)

        interaction = (
            user_vector * movie_vector
        ).sum(dim=1)

        user_bias = self.user_bias(users).squeeze()
        movie_bias = self.movie_bias(movies).squeeze()

        prediction = (
            self.global_bias
            + user_bias
            + movie_bias
            + interaction
        )

        return prediction

In [ ]:
num_users = len(user_id_to_index)
num_movies = len(movie_id_to_index)

global_mean = train_ratings["rating"].mean()

model = MatrixFactorization(
    num_users=num_users,
    num_movies=num_movies,
    global_mean=global_mean,
    embedding_dim=50
)

print(model)

## Model Training

The model is trained using a 2-million-rating sample from the training set.

The sample is used to reduce computational cost while keeping the training procedure practical on CPU hardware.

In [ ]:
sample_size = 2_000_000

train_sample = train_ratings.sample(
    n=sample_size,
    random_state=42
)

print("Training samples:", len(train_sample))

In [ ]:
train_users_sample = torch.tensor(
    train_sample["user_id"].map(user_id_to_index).values,
    dtype=torch.long
)

train_movies_sample = torch.tensor(
    train_sample["movie_id"].map(movie_id_to_index).values,
    dtype=torch.long
)

train_values_sample = torch.tensor(
    train_sample["rating"].values,
    dtype=torch.float32
)

In [ ]:
train_dataset_sample = TensorDataset(
    train_users_sample,
    train_movies_sample,
    train_values_sample
)
train_loader_sample = DataLoader(
    train_dataset_sample,
    batch_size=8192,
    shuffle=True
)

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [ ]:
model.train()

for epoch in range(3):

    total_loss = 0

    for users, movies, ratings_batch in train_loader_sample:

        optimizer.zero_grad()

        predictions = model(users, movies)

        loss = criterion(
            predictions,
            ratings_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader_sample)

    print(
        f"Epoch {epoch + 1}/3 - "
        f"Training Loss: {average_loss:.4f}"
    )

In [ ]:
model.eval()

with torch.no_grad():

    validation_users = torch.tensor(
        validation_ratings["user_id"].map(user_id_to_index).values,
        dtype=torch.long
    )

    validation_movies = torch.tensor(
        validation_ratings["movie_id"].map(movie_id_to_index).values,
        dtype=torch.long
    )

    validation_predictions = model(
        validation_users,
        validation_movies
    )

validation_predictions = validation_predictions.numpy()

## Collaborative Filtering Evaluation

The Matrix Factorization model is evaluated using MAE and RMSE on the validation set.

In [ ]:
cf_mae = mean_absolute_error(
    validation_ratings["rating"],
    validation_predictions
)

cf_rmse = np.sqrt(
    mean_squared_error(
        validation_ratings["rating"],
        validation_predictions
    )
)

print("CF MAE:", cf_mae)
print("CF RMSE:", cf_rmse)

In [ ]:
def predict_cf_scores(user_id, movie_ids):

    user_index = user_id_to_index[user_id]

    user_indices = torch.full(
        (len(movie_ids),),
        user_index,
        dtype=torch.long
    )

    movie_indices = torch.tensor(
        [
            movie_id_to_index[movie_id]
            for movie_id in movie_ids
        ],
        dtype=torch.long
    )

    model.eval()

    with torch.no_grad():
        predictions = model(
            user_indices,
            movie_indices
        )

    return predictions.numpy()

In [ ]:
user_seen_movies = (
    train_ratings
    .groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)

print("Users:", len(user_seen_movies))

In [ ]:
def get_unseen_movies(user_id):

    seen_movies = user_seen_movies.get(user_id, set())

    candidate_movies = movie_id_to_index.keys()

    unseen_movies = [
        movie_id
        for movie_id in candidate_movies
        if movie_id not in seen_movies
    ]

    return unseen_movies

In [ ]:
def recommend_cf(user_id, n=10):

    unseen_movies = get_unseen_movies(user_id)

    scores = predict_cf_scores(
        user_id,
        unseen_movies
    )

    recommendations = pd.DataFrame({
        "movie_id": unseen_movies,
        "cf_score": scores
    })

    recommendations = recommendations.sort_values(
        "cf_score",
        ascending=False
    ).head(n)

    recommendations = recommendations.merge(
        movies_content[["movie_id", "title", "genres"]],
        on="movie_id",
        how="left"
    )

    return recommendations[
        ["movie_id", "title", "genres", "cf_score"]
    ]

In [ ]:
recommend_cf(1, n=10)

In [ ]:
def precision_recall_at_k(
    user_id,
    recommendations,
    k=10
):

    test_relevant = set(
        test_ratings[
            (test_ratings["user_id"] == user_id)
            & (test_ratings["rating"] >= 4.0)
        ]["movie_id"]
    )

    if len(test_relevant) == 0:
        return None, None

    recommended_movies = set(
        recommendations.head(k)["movie_id"]
    )

    hits = len(
        recommended_movies & test_relevant
    )

    precision = hits / k

    recall = hits / len(test_relevant)

    return precision, recall

## Content-Based Recommendation

The Content-Based recommender represents each movie using its genres and user-generated tags.

TF-IDF is used to transform the combined movie metadata into feature vectors. A user preference profile is then constructed from movies rated highly by the user, and cosine similarity is used to rank candidate movies.

In [ ]:
tags = pd.read_csv(
    "../data/raw/ml-25m/tags.csv"
)

print("Tags shape:", tags.shape)

print("\nColumns:")
print(tags.columns.tolist())

print("\nMissing values:")
print(tags.isnull().sum())

print("\nUnique movies with tags:")
print(tags["movieId"].nunique())

print("\nUnique tags:")
print(tags["tag"].nunique())

print("\nSample:")
tags.head(20)

In [ ]:
tags_clean = tags.dropna(
    subset=["tag"]
).copy()

movie_tags = (
    tags_clean
    .groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

print("Movies with tags:", len(movie_tags))

print("\nColumns:")
print(movie_tags.columns.tolist())

print("\nSample:")
print(movie_tags.head(10))

In [ ]:
content_features = movies_content[
    ["movie_id", "title", "genres"]
].copy()

content_features = content_features.merge(
    movie_tags,
    left_on="movie_id",
    right_on="movieId",
    how="left"
)

content_features = content_features.drop(
    columns=["movieId"]
)

content_features["tag"] = (
    content_features["tag"]
    .fillna("")
)

content_features["genres"] = (
    content_features["genres"]
    .fillna("")
)

print("Shape:", content_features.shape)

print("\nMissing values:")
print(
    content_features[
        ["genres", "tag"]
    ].isnull().sum()
)

print("\nSample:")
content_features.head(10)

In [ ]:
content_features["combined_text"] = (
    content_features["genres"]
    .str.replace("|", " ", regex=False)
    + " "
    + content_features["tag"]
)

print(
    content_features[
        ["movie_id", "genres", "tag", "combined_text"]
    ].head(5)
)

In [ ]:
content_vectorizer = TfidfVectorizer(
    min_df=2,
    stop_words="english"
)

content_matrix = content_vectorizer.fit_transform(
    content_features["combined_text"]
)

print("Matrix shape:", content_matrix.shape)
print("Number of features:", len(
    content_vectorizer.get_feature_names_out()
))

In [ ]:
content_movie_index = {
    movie_id: index
    for index, movie_id in enumerate(
        content_features["movie_id"]
    )
}

print(
    "Number of movies in index:",
    len(content_movie_index)
)

print(
    "Movie 1 index:",
    content_movie_index[1]
)

## User Preference Profile

A user preference profile is constructed by combining the TF-IDF representations of movies rated by the user.

Higher-rated movies contribute more strongly to the resulting profile.

In [ ]:
def get_user_tfidf_profile(user_id):

    user_ratings = train_ratings[
        train_ratings["user_id"] == user_id
    ][["movie_id", "rating"]].copy()

    if len(user_ratings) == 0:
        return None

    # Map movie IDs to TF-IDF matrix rows
    user_ratings["content_index"] = (
        user_ratings["movie_id"]
        .map(content_movie_index)
    )

    # Keep only movies available in the content matrix
    user_ratings = user_ratings.dropna(
        subset=["content_index"]
    )

    if len(user_ratings) == 0:
        return None

    movie_indices = (
        user_ratings["content_index"]
        .astype(int)
        .values
    )

    movie_vectors = content_matrix[
        movie_indices
    ]

    ratings = user_ratings["rating"].values

    # Center ratings around neutral rating
    rating_weights = ratings - 3.5

    denominator = np.sum(
        np.abs(rating_weights)
    )

    if denominator == 0:
        return None

    # Weighted average of movie TF-IDF vectors
    weighted_vectors = (
        movie_vectors.multiply(
            rating_weights[:, np.newaxis]
        )
    )

    user_profile = (
        weighted_vectors.sum(axis=0)
        / denominator
    )

    return user_profile

In [ ]:
def get_user_tfidf_scores(user_id, candidate_movies):

    user_profile = get_user_tfidf_profile(user_id)

    if user_profile is None:
        return np.zeros(len(candidate_movies))

    candidate_indices = [
        content_movie_index[movie_id]
        for movie_id in candidate_movies
        if movie_id in content_movie_index
    ]

    candidate_vectors = content_matrix[
        candidate_indices
    ]

    # Convert numpy.matrix to regular 2D numpy array
    user_profile = np.asarray(
        user_profile
    )

    scores = cosine_similarity(
        candidate_vectors,
        user_profile
    ).ravel()

    return scores

## Content-Based Recommendation

Movies already rated by the user are excluded from the candidate set.

The remaining movies are ranked according to their cosine similarity with the user's content profile.

In [ ]:
def recommend_content(user_id, n=10):

    unseen_movies = get_unseen_movies(user_id)

    content_scores = get_user_tfidf_scores(
        user_id,
        unseen_movies
    )

    recommendations = pd.DataFrame({
        "movie_id": unseen_movies,
        "content_score": content_scores
    })

    recommendations = recommendations.sort_values(
        "content_score",
        ascending=False
    ).head(n)

    recommendations = recommendations.merge(
        movies_content[
            ["movie_id", "title", "genres"]
        ],
        on="movie_id",
        how="left"
    )

    return recommendations[
        [
            "movie_id",
            "title",
            "genres",
            "content_score"
        ]
    ]

In [ ]:
recommend_content(1, n=10)

In [ ]:
def precision_recall_at_k_validation(
    user_id,
    recommendations,
    k=10
):

    validation_relevant = set(
        validation_ratings[
            (validation_ratings["user_id"] == user_id)
            & (validation_ratings["rating"] >= 4.0)
        ]["movie_id"]
    )

    if len(validation_relevant) == 0:
        return None, None

    recommended_movies = set(
        recommendations.head(k)["movie_id"]
    )

    hits = len(
        recommended_movies & validation_relevant
    )

    precision = hits / k
    recall = hits / len(validation_relevant)

    return precision, recall

In [ ]:
validation_users = (
    validation_ratings["user_id"]
    .unique()
)

validation_users = validation_users[:30]

print("Validation users:", len(validation_users))

## Hybrid Recommendation and Hyperparameter Tuning

The Hybrid recommender combines Collaborative Filtering and Content-Based scores.

The parameter `alpha` controls the contribution of Collaborative Filtering:

- `alpha = 1.0`: Collaborative Filtering only
- `alpha = 0.0`: Content-Based Filtering only
- intermediate values: Hybrid recommendation

The value of `alpha` is selected using the validation set based on Top-N recommendation performance.

In [ ]:
alphas = np.arange(0.0, 1.1, 0.1)
alpha_results = []

for user_id in validation_users:

    unseen_movies = get_unseen_movies(user_id)

    # Calculate each model's scores only once
    cf_scores = predict_cf_scores(
        user_id,
        unseen_movies
    )

    content_scores = get_user_tfidf_scores(
        user_id,
        unseen_movies
    )

    # Normalize CF scores
    cf_min = cf_scores.min()
    cf_max = cf_scores.max()

    if cf_max > cf_min:
        cf_normalized = (
            (cf_scores - cf_min)
            / (cf_max - cf_min)
        )
    else:
        cf_normalized = np.zeros_like(cf_scores)

    for alpha in alphas:

        hybrid_scores = (
            alpha * cf_normalized
            + (1 - alpha) * content_scores
        )

        recommendations = pd.DataFrame({
            "movie_id": unseen_movies,
            "hybrid_score": hybrid_scores
        })

        recommendations = recommendations.sort_values(
            "hybrid_score",
            ascending=False
        ).head(10)

        precision, recall = (
            precision_recall_at_k_validation(
                user_id,
                recommendations,
                k=10
            )
        )

        if precision is not None:

            alpha_results.append({
                "user_id": user_id,
                "alpha": alpha,
                "precision": precision,
                "recall": recall
            })

    print(f"User {user_id} completed")

In [ ]:
alpha_results_df = pd.DataFrame(alpha_results)

alpha_summary = (
    alpha_results_df
    .groupby("alpha")[["precision", "recall"]]
    .mean()
    .reset_index()
)

alpha_summary

In [ ]:
test_predictions_cf = model(
    torch.tensor(
        test_ratings["user_id"]
        .map(user_id_to_index)
        .values,
        dtype=torch.long
    ),
    torch.tensor(
        test_ratings["movie_id"]
        .map(movie_id_to_index)
        .values,
        dtype=torch.long
    )
).detach().numpy()

test_actual_cf = test_ratings["rating"].values

cf_test_mae = mean_absolute_error(
    test_actual_cf,
    test_predictions_cf
)

cf_test_rmse = np.sqrt(
    mean_squared_error(
        test_actual_cf,
        test_predictions_cf
    )
)

print("CF Test MAE:", cf_test_mae)
print("CF Test RMSE:", cf_test_rmse)

## Random Split Evaluation

After model development and hyperparameter selection, the recommendation models are evaluated on the held-out test set.

Top-N recommendation performance is measured using Precision@10 and Recall@10.

A rating of 4.0 or higher is treated as a relevant recommendation.

In [ ]:
test_users = (
    test_ratings["user_id"]
    .unique()[:100]
)

print("Test users:", len(test_users))

In [ ]:
test_results = []

for user_id in test_users:

    unseen_movies = get_unseen_movies(user_id)

    # -------------------------
    # CF
    # -------------------------

    cf_scores = predict_cf_scores(
        user_id,
        unseen_movies
    )

    cf_recommendations = pd.DataFrame({
        "movie_id": unseen_movies,
        "score": cf_scores
    })

    cf_recommendations = (
        cf_recommendations
        .sort_values("score", ascending=False)
        .head(10)
    )

    cf_precision, cf_recall = (
        precision_recall_at_k(
            user_id,
            cf_recommendations,
            k=10
        )
    )

    # -------------------------
    # Content-Based
    # -------------------------

    content_scores = get_user_tfidf_scores(
        user_id,
        unseen_movies
    )

    content_recommendations = pd.DataFrame({
        "movie_id": unseen_movies,
        "score": content_scores
    })

    content_recommendations = (
        content_recommendations
        .sort_values("score", ascending=False)
        .head(10)
    )

    content_precision, content_recall = (
        precision_recall_at_k(
            user_id,
            content_recommendations,
            k=10
        )
    )

    # -------------------------
    # Hybrid
    # -------------------------

    cf_min = cf_scores.min()
    cf_max = cf_scores.max()

    if cf_max > cf_min:
        cf_normalized = (
            (cf_scores - cf_min)
            / (cf_max - cf_min)
        )
    else:
        cf_normalized = np.zeros_like(cf_scores)

    hybrid_scores = (
        0.7 * cf_normalized
        + 0.3 * content_scores
    )

    hybrid_recommendations = pd.DataFrame({
        "movie_id": unseen_movies,
        "score": hybrid_scores
    })

    hybrid_recommendations = (
        hybrid_recommendations
        .sort_values("score", ascending=False)
        .head(10)
    )

    hybrid_precision, hybrid_recall = (
        precision_recall_at_k(
            user_id,
            hybrid_recommendations,
            k=10
        )
    )

    # -------------------------
    # Save results
    # -------------------------

    if cf_precision is not None:
        test_results.append({
            "user_id": user_id,
            "cf_precision": cf_precision,
            "cf_recall": cf_recall,
            "content_precision": content_precision,
            "content_recall": content_recall,
            "hybrid_precision": hybrid_precision,
            "hybrid_recall": hybrid_recall
        })

    print(f"User {user_id} completed")

In [ ]:
test_results_df = pd.DataFrame(
    test_results
)

print(
    "Evaluated Users:",
    len(test_results_df)
)

print("\nMean Precision@10:")
print(
    "CF:",
    test_results_df["cf_precision"].mean()
)

print(
    "Content:",
    test_results_df["content_precision"].mean()
)

print(
    "Hybrid:",
    test_results_df["hybrid_precision"].mean()
)

print("\nMean Recall@10:")
print(
    "CF:",
    test_results_df["cf_recall"].mean()
)

print(
    "Content:",
    test_results_df["content_recall"].mean()
)

print(
    "Hybrid:",
    test_results_df["hybrid_recall"].mean()
)

## Temporal Evaluation

A chronological evaluation is used to simulate a real recommendation scenario in which models are trained using historical interactions and evaluated on future interactions.

The temporal split is defined as:

- Before 2018: Training
- 2018: Validation
- 2019 onward: Test

Only users and movies available during the corresponding training period are considered for the warm-start evaluation.

In [ ]:
# Load timestamp only from the original MovieLens ratings file

ratings_time = pd.read_csv(
    "../data/raw/ml-25m/ratings.csv",
    usecols=["userId", "movieId", "timestamp"]
)

ratings_time["datetime"] = pd.to_datetime(
    ratings_time["timestamp"],
    unit="s"
)

print("Overall rating period:")
print(
    ratings_time["datetime"].min(),
    "to",
    ratings_time["datetime"].max()
)

In [ ]:
# Prepare timestamp lookup
ratings_time_lookup = ratings_time[
    ["userId", "movieId", "datetime"]
].rename(
    columns={
        "userId": "user_id",
        "movieId": "movie_id"
    }
)

# Get timestamp for each split
train_time = train_ratings.merge(
    ratings_time_lookup,
    on=["user_id", "movie_id"],
    how="left"
)

validation_time = validation_ratings.merge(
    ratings_time_lookup,
    on=["user_id", "movie_id"],
    how="left"
)

test_time = test_ratings.merge(
    ratings_time_lookup,
    on=["user_id", "movie_id"],
    how="left"
)

# Display time ranges
print("Train:")
print(
    train_time["datetime"].min(),
    "to",
    train_time["datetime"].max()
)

print("\nValidation:")
print(
    validation_time["datetime"].min(),
    "to",
    validation_time["datetime"].max()
)

print("\nTest:")
print(
    test_time["datetime"].min(),
    "to",
    test_time["datetime"].max()
)

In [ ]:
# Create temporal splits from the original MovieLens ratings

ratings_temporal = ratings_time.copy()

train_temporal = ratings_temporal[
    ratings_temporal["datetime"] < "2018-01-01"
].copy()

validation_temporal = ratings_temporal[
    (ratings_temporal["datetime"] >= "2018-01-01")
    & (ratings_temporal["datetime"] < "2019-01-01")
].copy()

test_temporal = ratings_temporal[
    ratings_temporal["datetime"] >= "2019-01-01"
].copy()

print("Train:", len(train_temporal))
print("Validation:", len(validation_temporal))
print("Test:", len(test_temporal))

In [ ]:
print("\nTrain period:")
print(
    train_temporal["datetime"].min(),
    "to",
    train_temporal["datetime"].max()
)

print("\nValidation period:")
print(
    validation_temporal["datetime"].min(),
    "to",
    validation_temporal["datetime"].max()
)

print("\nTest period:")
print(
    test_temporal["datetime"].min(),
    "to",
    test_temporal["datetime"].max()
)

In [ ]:
train_temporal_users = set(
    train_temporal["userId"].unique()
)

train_temporal_movies = set(
    train_temporal["movieId"].unique()
)

validation_users = set(
    validation_temporal["userId"].unique()
)

validation_movies = set(
    validation_temporal["movieId"].unique()
)

test_users = set(
    test_temporal["userId"].unique()
)

test_movies = set(
    test_temporal["movieId"].unique()
)

print("Temporal Train users:", len(train_temporal_users))
print("Temporal Train movies:", len(train_temporal_movies))

print("\nValidation users:", len(validation_users))
print(
    "Validation users seen in Train:",
    len(validation_users & train_temporal_users)
)

print("\nValidation movies:", len(validation_movies))
print(
    "Validation movies seen in Train:",
    len(validation_movies & train_temporal_movies)
)

print("\nTest users:", len(test_users))
print(
    "Test users seen in Train:",
    len(test_users & train_temporal_users)
)

print("\nTest movies:", len(test_movies))
print(
    "Test movies seen in Train:",
    len(test_movies & train_temporal_movies)
)

In [ ]:
# Warm-start validation ratings
validation_warm = validation_temporal[
    validation_temporal["userId"].isin(train_temporal_users)
].copy()

# Warm-start test ratings
test_warm = test_temporal[
    test_temporal["userId"].isin(train_temporal_users)
].copy()


print("Validation total ratings:", len(validation_temporal))
print("Validation warm-start ratings:", len(validation_warm))

print("\nTest total ratings:", len(test_temporal))
print("Test warm-start ratings:", len(test_warm))

In [ ]:
# Convert temporal splits to the same column format
# used by the CF model

train_temporal_cf = train_temporal[
    ["userId", "movieId"]
].rename(
    columns={
        "userId": "user_id",
        "movieId": "movie_id"
    }
).copy()

validation_temporal_cf = validation_warm[
    ["userId", "movieId"]
].rename(
    columns={
        "userId": "user_id",
        "movieId": "movie_id"
    }
).copy()

test_temporal_cf = test_warm[
    ["userId", "movieId"]
].rename(
    columns={
        "userId": "user_id",
        "movieId": "movie_id"
    }
).copy()


print("Temporal CF train:", train_temporal_cf.shape)
print("Temporal CF validation:", validation_temporal_cf.shape)
print("Temporal CF test:", test_temporal_cf.shape)

### Temporal Content-Based Recommendation

To reduce future-information leakage, only user-generated tags created before 2018 are used to construct the temporal content features.

Movie genres are treated as static metadata.

In [ ]:
# Load ratings with both timestamp and rating
ratings_temporal_full = pd.read_csv(
    "../data/raw/ml-25m/ratings.csv",
    usecols=["userId", "movieId", "rating", "timestamp"]
)

ratings_temporal_full["datetime"] = pd.to_datetime(
    ratings_temporal_full["timestamp"],
    unit="s"
)


# Temporal splits
train_temporal = ratings_temporal_full[
    ratings_temporal_full["datetime"] < "2018-01-01"
].copy()

validation_temporal = ratings_temporal_full[
    (ratings_temporal_full["datetime"] >= "2018-01-01")
    & (ratings_temporal_full["datetime"] < "2019-01-01")
].copy()

test_temporal = ratings_temporal_full[
    ratings_temporal_full["datetime"] >= "2019-01-01"
].copy()


print("Train:", train_temporal.shape)
print("Validation:", validation_temporal.shape)
print("Test:", test_temporal.shape)

In [ ]:
# Get users and movies from temporal training data

temporal_user_ids = train_temporal["userId"].unique()
temporal_movie_ids = train_temporal["movieId"].unique()


# User ID -> embedding index
temporal_user_id_to_index = {
    user_id: index
    for index, user_id in enumerate(temporal_user_ids)
}


# Movie ID -> embedding index
temporal_movie_id_to_index = {
    movie_id: index
    for index, movie_id in enumerate(temporal_movie_ids)
}


print("Temporal Train users:", len(temporal_user_id_to_index))
print("Temporal Train movies:", len(temporal_movie_id_to_index))

In [ ]:
sample_size_temporal = 2_000_000

train_temporal_sample = train_temporal.sample(
    n=sample_size_temporal,
    random_state=42
).copy()

print(
    "Temporal training sample:",
    train_temporal_sample.shape
)

In [ ]:
train_users_temporal = torch.tensor(
    train_temporal_sample["userId"]
    .map(temporal_user_id_to_index)
    .values,
    dtype=torch.long
)

train_movies_temporal = torch.tensor(
    train_temporal_sample["movieId"]
    .map(temporal_movie_id_to_index)
    .values,
    dtype=torch.long
)

train_ratings_temporal = torch.tensor(
    train_temporal_sample["rating"].values,
    dtype=torch.float32
)


train_dataset_temporal = TensorDataset(
    train_users_temporal,
    train_movies_temporal,
    train_ratings_temporal
)


train_loader_temporal = DataLoader(
    train_dataset_temporal,
    batch_size=8192,
    shuffle=True
)


print("Users tensor:", train_users_temporal.shape)
print("Movies tensor:", train_movies_temporal.shape)
print("Ratings tensor:", train_ratings_temporal.shape)
print("Number of batches:", len(train_loader_temporal))

In [ ]:
temporal_global_mean = train_temporal_sample["rating"].mean()

temporal_model = MatrixFactorization(
    num_users=len(temporal_user_id_to_index),
    num_movies=len(temporal_movie_id_to_index),
    global_mean=temporal_global_mean,
    embedding_dim=50
)

temporal_criterion = nn.MSELoss()

temporal_optimizer = torch.optim.Adam(
    temporal_model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)


print("Temporal global mean:", temporal_global_mean)
print(temporal_model)

In [ ]:
num_epochs_temporal = 3

temporal_model.train()

for epoch in range(num_epochs_temporal):

    total_loss = 0.0

    for users, movies_batch, ratings_batch in train_loader_temporal:

        temporal_optimizer.zero_grad()

        predictions = temporal_model(
            users,
            movies_batch
        )

        loss = temporal_criterion(
            predictions,
            ratings_batch
        )

        loss.backward()
        temporal_optimizer.step()

        total_loss += loss.item()

    average_loss = (
        total_loss
        / len(train_loader_temporal)
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs_temporal} "
        f"- Loss: {average_loss:.4f}"
    )

In [ ]:
# Keep only validation ratings whose users and movies
# were seen during temporal training

validation_temporal_eval = validation_temporal[
    validation_temporal["userId"].isin(
        temporal_user_id_to_index
    )
    &
    validation_temporal["movieId"].isin(
        temporal_movie_id_to_index
    )
].copy()


validation_users_tensor = torch.tensor(
    validation_temporal_eval["userId"]
    .map(temporal_user_id_to_index)
    .values,
    dtype=torch.long
)

validation_movies_tensor = torch.tensor(
    validation_temporal_eval["movieId"]
    .map(temporal_movie_id_to_index)
    .values,
    dtype=torch.long
)

validation_ratings_tensor = torch.tensor(
    validation_temporal_eval["rating"].values,
    dtype=torch.float32
)


print(
    "Validation ratings:",
    len(validation_temporal_eval)
)

In [ ]:
temporal_model.eval()

with torch.no_grad():

    validation_predictions = temporal_model(
        validation_users_tensor,
        validation_movies_tensor
    ).numpy()


validation_actual = (
    validation_ratings_tensor.numpy()
)


validation_mae = mean_absolute_error(
    validation_actual,
    validation_predictions
)

validation_rmse = np.sqrt(
    mean_squared_error(
        validation_actual,
        validation_predictions
    )
)


print(
    f"Temporal Validation MAE: {validation_mae:.4f}"
)

print(
    f"Temporal Validation RMSE: {validation_rmse:.4f}"
)

In [ ]:
test_temporal_eval = test_temporal[
    test_temporal["userId"].isin(
        temporal_user_id_to_index
    )
    &
    test_temporal["movieId"].isin(
        temporal_movie_id_to_index
    )
].copy()


test_users_tensor = torch.tensor(
    test_temporal_eval["userId"]
    .map(temporal_user_id_to_index)
    .values,
    dtype=torch.long
)

test_movies_tensor = torch.tensor(
    test_temporal_eval["movieId"]
    .map(temporal_movie_id_to_index)
    .values,
    dtype=torch.long
)

test_ratings_tensor = torch.tensor(
    test_temporal_eval["rating"].values,
    dtype=torch.float32
)


print(
    "Temporal Test ratings:",
    len(test_temporal_eval)
)

In [ ]:
temporal_model.eval()

with torch.no_grad():

    test_predictions = temporal_model(
        test_users_tensor,
        test_movies_tensor
    ).numpy()


test_actual = test_ratings_tensor.numpy()


test_mae = mean_absolute_error(
    test_actual,
    test_predictions
)

test_rmse = np.sqrt(
    mean_squared_error(
        test_actual,
        test_predictions
    )
)


print(
    f"Temporal Test MAE: {test_mae:.4f}"
)

print(
    f"Temporal Test RMSE: {test_rmse:.4f}"
)

In [ ]:
temporal_user_seen_movies = (
    train_temporal
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print(
    "Users with seen-movie history:",
    len(temporal_user_seen_movies)
)

In [ ]:
def get_temporal_unseen_movies(user_id):

    seen_movies = temporal_user_seen_movies.get(
        user_id,
        set()
    )

    candidate_movies = temporal_movie_id_to_index.keys()

    unseen_movies = [
        movie_id
        for movie_id in candidate_movies
        if movie_id not in seen_movies
    ]

    return unseen_movies

In [ ]:
def predict_temporal_cf_scores(user_id, movie_ids):

    user_index = temporal_user_id_to_index[user_id]

    user_indices = torch.full(
        (len(movie_ids),),
        user_index,
        dtype=torch.long
    )

    movie_indices = torch.tensor(
        [
            temporal_movie_id_to_index[movie_id]
            for movie_id in movie_ids
        ],
        dtype=torch.long
    )

    temporal_model.eval()

    with torch.no_grad():

        predictions = temporal_model(
            user_indices,
            movie_indices
        )

    return predictions.numpy()

In [ ]:
def recommend_temporal_cf(user_id, n=10):

    unseen_movies = get_temporal_unseen_movies(
        user_id
    )

    scores = predict_temporal_cf_scores(
        user_id,
        unseen_movies
    )

    recommendations = pd.DataFrame({
        "movie_id": unseen_movies,
        "cf_score": scores
    })

    recommendations = recommendations.sort_values(
        "cf_score",
        ascending=False
    ).head(n)

    recommendations = recommendations.merge(
        movies_content[
            ["movie_id", "title", "genres"]
        ],
        on="movie_id",
        how="left"
    )

    return recommendations[
        [
            "movie_id",
            "title",
            "genres",
            "cf_score"
        ]
    ]

In [ ]:
# Relevant movies in temporal validation
# Rating >= 4.0 is considered relevant

validation_relevant = (
    validation_temporal_eval[
        validation_temporal_eval["rating"] >= 4.0
    ]
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print(
    "Validation users with relevant movies:",
    len(validation_relevant)
)

In [ ]:
evaluation_users_temporal = list(
    validation_relevant.keys()
)

evaluation_users_temporal = (
    pd.Series(evaluation_users_temporal)
    .sample(
        n=100,
        random_state=42
    )
    .tolist()
)

print(
    "Temporal Validation evaluation users:",
    len(evaluation_users_temporal)
)

In [ ]:
precisions_temporal = []
recalls_temporal = []

for user_id in evaluation_users_temporal:

    recommendations = recommend_temporal_cf(
        user_id,
        n=10
    )

    recommended_movies = set(
        recommendations["movie_id"]
    )

    relevant_movies = validation_relevant[
        user_id
    ]

    hits = len(
        recommended_movies & relevant_movies
    )

    precision = hits / 10

    recall = (
        hits / len(relevant_movies)
        if len(relevant_movies) > 0
        else 0
    )

    precisions_temporal.append(precision)
    recalls_temporal.append(recall)


mean_precision_temporal = np.mean(
    precisions_temporal
)

mean_recall_temporal = np.mean(
    recalls_temporal
)


print(
    f"Temporal Validation Precision@10: "
    f"{mean_precision_temporal:.4f}"
)

print(
    f"Temporal Validation Recall@10: "
    f"{mean_recall_temporal:.4f}"
)

In [ ]:
# Relevant movies in temporal test
# Rating >= 4.0 is considered relevant

test_relevant = (
    test_temporal_eval[
        test_temporal_eval["rating"] >= 4.0
    ]
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print(
    "Test users with relevant movies:",
    len(test_relevant)
)

In [ ]:
evaluation_users_test_temporal = list(
    test_relevant.keys()
)

evaluation_users_test_temporal = (
    pd.Series(evaluation_users_test_temporal)
    .sample(
        n=100,
        random_state=42
    )
    .tolist()
)

print(
    "Temporal Test evaluation users:",
    len(evaluation_users_test_temporal)
)

In [ ]:
precisions_test_temporal = []
recalls_test_temporal = []

for user_id in evaluation_users_test_temporal:

    recommendations = recommend_temporal_cf(
        user_id,
        n=10
    )

    recommended_movies = set(
        recommendations["movie_id"]
    )

    relevant_movies = test_relevant[
        user_id
    ]

    hits = len(
        recommended_movies & relevant_movies
    )

    precision = hits / 10

    recall = (
        hits / len(relevant_movies)
        if len(relevant_movies) > 0
        else 0
    )

    precisions_test_temporal.append(
        precision
    )

    recalls_test_temporal.append(
        recall
    )


mean_precision_test_temporal = np.mean(
    precisions_test_temporal
)

mean_recall_test_temporal = np.mean(
    recalls_test_temporal
)


print(
    f"Temporal Test Precision@10: "
    f"{mean_precision_test_temporal:.4f}"
)

print(
    f"Temporal Test Recall@10: "
    f"{mean_recall_test_temporal:.4f}"
)

In [ ]:
def get_temporal_user_tfidf_profile(user_id):

    user_ratings = train_temporal[
        train_temporal["userId"] == user_id
    ][
        ["movieId", "rating"]
    ].copy()

    if len(user_ratings) == 0:
        return None

    user_ratings["content_index"] = (
        user_ratings["movieId"]
        .map(content_movie_index)
    )

    user_ratings = user_ratings.dropna(
        subset=["content_index"]
    )

    if len(user_ratings) == 0:
        return None

    movie_indices = (
        user_ratings["content_index"]
        .astype(int)
        .values
    )

    movie_vectors = content_matrix[
        movie_indices
    ]

    ratings = user_ratings["rating"].values

    rating_weights = ratings - 3.5

    denominator = np.sum(
        np.abs(rating_weights)
    )

    if denominator == 0:
        return None

    weighted_vectors = (
        movie_vectors.multiply(
            rating_weights[:, np.newaxis]
        )
    )

    user_profile = (
        weighted_vectors.sum(axis=0)
        / denominator
    )

    return user_profile

In [ ]:
def get_temporal_user_tfidf_scores(
    user_id,
    candidate_movies
):

    user_profile = (
        get_temporal_user_tfidf_profile(
            user_id
        )
    )

    if user_profile is None:
        return np.zeros(
            len(candidate_movies)
        )

    candidate_indices = [
        content_movie_index[movie_id]
        for movie_id in candidate_movies
        if movie_id in content_movie_index
    ]

    candidate_vectors = content_matrix[
        candidate_indices
    ]

    user_profile = np.asarray(
        user_profile
    )

    scores = cosine_similarity(
        candidate_vectors,
        user_profile
    ).ravel()

    return scores

In [ ]:
def recommend_temporal_content(user_id, n=10):

    unseen_movies = get_temporal_unseen_movies(
        user_id
    )

    content_scores = (
        get_temporal_user_tfidf_scores(
            user_id,
            unseen_movies
        )
    )

    recommendations = pd.DataFrame({
        "movie_id": unseen_movies,
        "content_score": content_scores
    })

    recommendations = recommendations.sort_values(
        "content_score",
        ascending=False
    ).head(n)

    recommendations = recommendations.merge(
        movies_content[
            ["movie_id", "title", "genres"]
        ],
        on="movie_id",
        how="left"
    )

    return recommendations[
        [
            "movie_id",
            "title",
            "genres",
            "content_score"
        ]
    ]

In [ ]:
test_user = next(iter(temporal_user_id_to_index))
recommend_temporal_content(test_user, n=10)

In [ ]:
tags_temporal = tags.copy()

tags_temporal["datetime"] = pd.to_datetime(
    tags_temporal["timestamp"],
    unit="s"
)

tags_temporal_train = tags_temporal[
    tags_temporal["datetime"] < "2018-01-01"
].copy()

print(
    "Original tags:",
    len(tags)
)

print(
    "Pre-2018 tags:",
    len(tags_temporal_train)
)

print(
    "Unique movies with pre-2018 tags:",
    tags_temporal_train["movieId"].nunique()
)

print(
    "Unique tags:",
    tags_temporal_train["tag"].nunique()
)

In [ ]:
tags_temporal_clean = tags_temporal_train.dropna(
    subset=["tag"]
).copy()

temporal_movie_tags = (
    tags_temporal_clean
    .groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

temporal_content_features = movies_content[
    ["movie_id", "title", "genres"]
].copy()

temporal_content_features = temporal_content_features.merge(
    temporal_movie_tags,
    left_on="movie_id",
    right_on="movieId",
    how="left"
)

temporal_content_features = temporal_content_features.drop(
    columns=["movieId"]
)

temporal_content_features["tag"] = (
    temporal_content_features["tag"]
    .fillna("")
)

temporal_content_features["genres"] = (
    temporal_content_features["genres"]
    .fillna("")
)

temporal_content_features["combined_text"] = (
    temporal_content_features["genres"]
    .str.replace("|", " ", regex=False)
    + " "
    + temporal_content_features["tag"]
)

print(
    "Temporal content features shape:",
    temporal_content_features.shape
)

print(
    "Missing tags:",
    temporal_content_features["tag"].isna().sum()
)

print(
    "Movies with tags:",
    (
        temporal_content_features["tag"].str.len() > 0
    ).sum()
)

In [ ]:
temporal_content_vectorizer = TfidfVectorizer(
    min_df=2,
    stop_words="english"
)

temporal_content_matrix = (
    temporal_content_vectorizer.fit_transform(
        temporal_content_features["combined_text"]
    )
)

print(
    "Temporal TF-IDF matrix shape:",
    temporal_content_matrix.shape
)

print(
    "Number of features:",
    len(
        temporal_content_vectorizer.get_feature_names_out()
    )
)

In [ ]:
temporal_content_movie_index = {
    movie_id: index
    for index, movie_id in enumerate(
        temporal_content_features["movie_id"]
    )
}

print(
    "Number of mapped movies:",
    len(temporal_content_movie_index)
)

print(
    "Index of movie 1:",
    temporal_content_movie_index.get(1)
)

In [ ]:
def get_temporal_user_tfidf_scores(
    user_id,
    candidate_movies
):

    user_profile = (
        get_temporal_user_tfidf_profile(
            user_id
        )
    )

    if user_profile is None:
        return np.zeros(
            len(candidate_movies)
        )

    candidate_indices = [
        temporal_content_movie_index[movie_id]
        for movie_id in candidate_movies
        if movie_id in temporal_content_movie_index
    ]

    candidate_vectors = temporal_content_matrix[
        candidate_indices
    ]

    user_profile = np.asarray(
        user_profile
    )

    scores = cosine_similarity(
        candidate_vectors,
        user_profile
    ).ravel()

    return scores

In [ ]:
def recommend_temporal_content(user_id, n=10):

    unseen_movies = get_temporal_unseen_movies(
        user_id
    )

    content_scores = (
        get_temporal_user_tfidf_scores(
            user_id,
            unseen_movies
        )
    )

    recommendations = pd.DataFrame({
        "movie_id": unseen_movies,
        "content_score": content_scores
    })

    recommendations = recommendations.sort_values(
        "content_score",
        ascending=False
    ).head(n)

    recommendations = recommendations.merge(
        temporal_content_features[
            ["movie_id", "title", "genres"]
        ],
        on="movie_id",
        how="left"
    )

    return recommendations[
        [
            "movie_id",
            "title",
            "genres",
            "content_score"
        ]
    ]

In [ ]:
precisions_content_test_temporal = []
recalls_content_test_temporal = []

for user_id in evaluation_users_test_temporal:

    recommendations = recommend_temporal_content(
        user_id,
        n=10
    )

    recommended_movies = set(
        recommendations["movie_id"]
    )

    relevant_movies = test_relevant[user_id]

    hits = len(
        recommended_movies & relevant_movies
    )

    precision = hits / 10

    recall = (
        hits / len(relevant_movies)
        if len(relevant_movies) > 0
        else 0
    )

    precisions_content_test_temporal.append(
        precision
    )

    recalls_content_test_temporal.append(
        recall
    )


mean_precision_content_test_temporal = np.mean(
    precisions_content_test_temporal
)

mean_recall_content_test_temporal = np.mean(
    recalls_content_test_temporal
)

print(
    f"Temporal Content Test Precision@10: "
    f"{mean_precision_content_test_temporal:.4f}"
)

print(
    f"Temporal Content Test Recall@10: "
    f"{mean_recall_content_test_temporal:.4f}"
)

### Temporal Hybrid Recommendation

The temporal Hybrid recommender combines the temporal Collaborative Filtering and Content-Based scores.

The same evaluation protocol is applied to the temporal test period to measure how well the combined approach generalizes to future user interactions.

In [ ]:
def recommend_temporal_hybrid(
    user_id,
    n=10,
    alpha=0.7
):

    unseen_movies = get_temporal_unseen_movies(
        user_id
    )

    cf_scores = predict_temporal_cf_scores(
        user_id,
        unseen_movies
    )

    content_scores = (
        get_temporal_user_tfidf_scores(
            user_id,
            unseen_movies
        )
    )

    # Normalize CF scores
    cf_min = cf_scores.min()
    cf_max = cf_scores.max()

    if cf_max > cf_min:
        cf_normalized = (
            (cf_scores - cf_min)
            / (cf_max - cf_min)
        )
    else:
        cf_normalized = np.zeros_like(
            cf_scores
        )

    hybrid_scores = (
        alpha * cf_normalized
        + (1 - alpha) * content_scores
    )

    recommendations = pd.DataFrame({
        "movie_id": unseen_movies,
        "cf_score": cf_scores,
        "content_score": content_scores,
        "hybrid_score": hybrid_scores
    })

    recommendations = recommendations.sort_values(
        "hybrid_score",
        ascending=False
    ).head(n)

    recommendations = recommendations.merge(
        temporal_content_features[
            ["movie_id", "title", "genres"]
        ],
        on="movie_id",
        how="left"
    )

    return recommendations[
        [
            "movie_id",
            "title",
            "genres",
            "cf_score",
            "content_score",
            "hybrid_score"
        ]
    ]

In [ ]:
recommend_temporal_hybrid(
    test_user,
    n=10,
    alpha=0.7
)

In [ ]:
precisions_hybrid_test_temporal = []
recalls_hybrid_test_temporal = []

for user_id in evaluation_users_test_temporal:

    recommendations = recommend_temporal_hybrid(
        user_id,
        n=10,
        alpha=0.7
    )

    recommended_movies = set(
        recommendations["movie_id"]
    )

    relevant_movies = test_relevant[user_id]

    hits = len(
        recommended_movies & relevant_movies
    )

    precision = hits / 10

    recall = (
        hits / len(relevant_movies)
        if len(relevant_movies) > 0
        else 0
    )

    precisions_hybrid_test_temporal.append(
        precision
    )

    recalls_hybrid_test_temporal.append(
        recall
    )


mean_precision_hybrid_test_temporal = np.mean(
    precisions_hybrid_test_temporal
)

mean_recall_hybrid_test_temporal = np.mean(
    recalls_hybrid_test_temporal
)

print(
    f"Temporal Hybrid Test Precision@10: "
    f"{mean_precision_hybrid_test_temporal:.4f}"
)

print(
    f"Temporal Hybrid Test Recall@10: "
    f"{mean_recall_hybrid_test_temporal:.4f}"
)

## Final Evaluation Summary

The following table summarizes the rating-prediction and Top-N recommendation performance under the random and temporal evaluation protocols.

In [ ]:
evaluation_summary = pd.DataFrame({
    "Evaluation": [
        "Random Split",
        "Random Split",
        "Random Split",
        "Temporal Split",
        "Temporal Split",
        "Temporal Split"
    ],
    "Model": [
        "CF",
        "Content-Based",
        "Hybrid",
        "CF",
        "Content-Based",
        "Hybrid"
    ],
    "MAE": [
        0.6939,
        np.nan,
        np.nan,
        0.6730,
        np.nan,
        np.nan
    ],
    "RMSE": [
        0.9012,
        np.nan,
        np.nan,
        0.9010,
        np.nan,
        np.nan
    ],
    "Precision@10": [
        0.0980,
        0.0566,
        0.1222,
        0.0450,
        0.0310,
        0.0530
    ],
    "Recall@10": [
        0.0417,
        0.0409,
        0.0579,
        0.0191,
        0.0176,
        0.0220
    ]
})

evaluation_summary

## Evaluation Notes and Limitations

- The random split provides a benchmark but may allow temporal information to appear across training and test sets.
- The temporal split provides a more realistic estimate of future recommendation performance.
- Collaborative Filtering was trained on a 2-million-rating sample because full training was computationally expensive on the available CPU hardware.
- Top-N evaluation uses Precision@10 and Recall@10, with ratings of 4.0 or higher treated as relevant.
- The temporal Top-N evaluation uses a sampled set of eligible users.
- Temporal Content-Based features use only pre-2018 tags to reduce future-information leakage.
- The reported metrics depend on the dataset, sampling strategy, split protocol, and evaluation design and should not be interpreted as universal real-world performance estimates.

## Conclusion

The notebook compared Collaborative Filtering, Content-Based Filtering, and Hybrid recommendation approaches under both random and temporal evaluation settings.

The temporal evaluation provides the more realistic estimate of how the recommender generalizes to future user interactions.